# EPIC — EXP033 LOW_TP Study

Следующий шаг после false-positive исследования.

Здесь **нет обучения основной модели**. Берём готовый:

```text
CNN-EXP-033
NO BPE
RF = 2053 bp
32-bp regional head
checkpoint = 10k
```

и исследуем настоящие положительные регионы, которые модель ранжирует низко.

## Главные вопросы

1. Какие TSS модель пропускает из TOP-10% регионов?
2. Связано ли это просто со слабым csRNA-seq signal?
3. Есть ли у LOW_TP другая DNA architecture?
4. Не лежат ли пропущенные TSS чаще на краях 32-bp bins?
5. Отличает ли простой sequence classifier HIGH_TP от LOW_TP?

Используем два вида групп:

```text
CAPTURED_TOP10
positive region попал в глобальные top-10% по score

MISSED_TOP10
positive region НЕ попал в top-10%

HIGH_TP
верхние 8000 positive regions по score

LOW_TP
нижние 8000 positive regions по score
```

`CAPTURED/MISSED` отвечает на практический вопрос recall.

`HIGH/LOW` даёт более контрастный материал для sequence diagnostics.


In [1]:
from pathlib import Path
import copy
import gc
import math
import sys
import time
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

import torch
import torch.nn as nn
import torch_directml

CURRENT_DIR = Path.cwd().resolve()

PROJECT_ROOT = next(
    candidate
    for candidate in (
        CURRENT_DIR,
        *CURRENT_DIR.parents,
    )
    if (candidate / "README.md").is_file()
    and (candidate / "src/ml_project").is_dir()
)

SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(
        0,
        str(SRC_DIR),
    )

from ml_project.epic_baseline import (
    load_baseline_sequences,
)

from ml_project.epic_cnn import (
    DirectMLAdam,
    ProfileWindowConfig,
    ProfileWindowGenerator,
    create_cnn_presence_intensity,
)

from ml_project.epic_data import (
    load_split,
)

print("Project:", PROJECT_ROOT)


Project: D:\Projects\EPIC\EPIC


In [2]:
device = torch_directml.device()

print("DirectML device:", device)
print("GPU:", torch_directml.device_name(0))


DirectML device: privateuseone:0
GPU: AMD Radeon RX 6750 GRE 12GB 


In [3]:
# ============================================================
# DATA / WINDOW CONFIG
# ============================================================

split = load_split(
    PROJECT_ROOT
)

sequences, fasta_alphabet = (
    load_baseline_sequences(
        split
    )
)

train_contigs = tuple(
    sorted(
        set(
            split.intervals(
                "train"
            )[
                "contig"
            ].astype(str)
        )
    )
)

validation_contigs = tuple(
    sorted(
        set(
            split.intervals(
                "validation"
            )[
                "contig"
            ].astype(str)
        )
    )
)

assert set(
    train_contigs
).isdisjoint(
    validation_contigs
)

# Keep 2048-bp target so the regional grid is directly comparable
# with the recent regional experiments.
TARGET_LENGTH = 2048
CONTEXT_FLANK = 130

REGION_WIDTHS = (
    32,
)

PRIMARY_WIDTH = 32

BLOCK_SETTINGS_RF2053 = (
    (7, 1),
    (3, 2),
    (3, 4),
    (3, 8),
    (3, 16),
    (3, 32),
    (3, 64),
    (3, 128),
    (3, 256),
)

BACKBONE_RF_BP = (
    1
    + sum(
        2 * (kernel_size - 1) * dilation
        for kernel_size, dilation
        in BLOCK_SETTINGS_RF2053
    )
)

REGIONAL_UNION_RF_BP = (
    BACKBONE_RF_BP
    + PRIMARY_WIDTH
    - 1
)

assert BACKBONE_RF_BP == 2053
assert REGIONAL_UNION_RF_BP == 2084

assert TARGET_LENGTH % 32 == 0

profile_config = (
    ProfileWindowConfig(
        target_length=TARGET_LENGTH,
        context_flank=CONTEXT_FLANK,
        positive_branch_fraction=0.5,
    )
)

train_generator = (
    ProfileWindowGenerator(
        prepared_split=split,
        sequences=sequences,
        part="train",
        config=profile_config,
        seed=42,
    )
)

validation_generator = (
    ProfileWindowGenerator(
        prepared_split=split,
        sequences=sequences,
        part="validation",
        config=profile_config,
        seed=42,
    )
)

display(split.summary)

print("Train contigs:", len(train_contigs))
print(
    "Validation contigs:",
    len(validation_contigs),
)
print(
    "Input length:",
    TARGET_LENGTH
    + 2 * CONTEXT_FLANK,
)
print(
    "Target length:",
    TARGET_LENGTH,
)
print(
    "Region widths:",
    REGION_WIDTHS,
)

print(
    "Backbone RF:",
    BACKBONE_RF_BP,
    "bp",
)

print(
    "32-bp regional union RF:",
    REGIONAL_UNION_RF_BP,
    "bp",
)


,split,contigs,intervals_both_strands,whitelist_bp,position_strand_count,percent_of_all_positions,positive_positions,negative_positions,positive_percent
0,train,9,168998,114032080,228064160,59.316784,170443.0,227893717.0,0.074735
1,validation,3,60386,41903743,83807486,21.797334,56205.0,83751281.0,0.067064
2,test,3,54846,36306695,72613390,18.885882,NaN,NaN,NaN


Train contigs: 9
Validation contigs: 3
Input length: 2308
Target length: 2048
Region widths: (32,)
Backbone RF: 2053 bp
32-bp regional union RF: 2084 bp


In [4]:
# ============================================================
# REGIONAL HEAD ON TOP OF THE EXISTING BASE-LEVEL BACKBONE
# ============================================================


class RegionalPoolHead(
    nn.Module
):
    def __init__(
        self,
        in_channels,
    ):
        super().__init__()

        self.classifier = (
            nn.Sequential(
                nn.Conv1d(
                    2 * in_channels,
                    32,
                    1,
                ),
                nn.GELU(),
                nn.Conv1d(
                    32,
                    1,
                    1,
                ),
            )
        )

    def forward(
        self,
        target_features,
        width,
    ):
        B, C, L = (
            target_features.shape
        )

        if L % width != 0:
            raise ValueError(
                (L, width)
            )

        n_regions = (
            L // width
        )

        x = (
            target_features
            .reshape(
                B,
                C,
                n_regions,
                width,
            )
        )

        mean_features = (
            x.mean(
                dim=-1
            )
        )

        # RMS instead of max:
        # DirectML max backward caused scatter fallback/errors
        # in earlier regional experiments.
        rms_features = torch.sqrt(
            (
                x * x
            ).mean(
                dim=-1
            )
            + 1e-6
        )

        pooled = torch.cat(
            (
                mean_features,
                rms_features,
            ),
            dim=1,
        ).contiguous()

        return self.classifier(
            pooled
        )


class BaseLevelRegionalModel(
    nn.Module
):
    def __init__(self):
        super().__init__()

        self.base = (
            create_cnn_presence_intensity(
                block_settings=(
                    BLOCK_SETTINGS_RF2053
                )
            )
        )

        if not hasattr(
            self.base,
            "profile_head",
        ):
            raise AttributeError(
                "Expected base model to expose .profile_head"
            )

        self.backbone_channels = int(
            self.base
            .profile_head
            .in_channels
        )


        assert (
            len(
                self.base
                .backbone
                .blocks
            )
            == len(
                BLOCK_SETTINGS_RF2053
            )
        )

        # Old per-base profile head is NOT part of the new objective.
        # Freeze its parameters; its input is captured by a hook.
        for parameter in (
            self.base
            .profile_head
            .parameters()
        ):
            parameter.requires_grad = False

        self.heads = nn.ModuleDict({
            str(width): RegionalPoolHead(
                self.backbone_channels
            )
            for width in REGION_WIDTHS
        })

        self._captured_features = None

        self._feature_hook = (
            self.base
            .profile_head
            .register_forward_pre_hook(
                self._capture_backbone_features
            )
        )

    def _capture_backbone_features(
        self,
        module,
        inputs,
    ):
        self._captured_features = (
            inputs[0]
        )

    def forward(
        self,
        x,
    ):
        self._captured_features = None

        # Run the known DirectML-safe base model.
        # Its old outputs are intentionally ignored.
        _ = self.base(
            x
        )

        features = (
            self._captured_features
        )

        self._captured_features = None

        if features is None:
            raise RuntimeError(
                "Backbone feature hook did not fire."
            )

        expected_length = (
            TARGET_LENGTH
            + 2 * CONTEXT_FLANK
        )

        if (
            features.shape[-1]
            != expected_length
        ):
            raise ValueError(
                (
                    features.shape,
                    expected_length,
                )
            )

        target_features = (
            features[
                :,
                :,
                CONTEXT_FLANK:
                CONTEXT_FLANK
                + TARGET_LENGTH
            ]
            .contiguous()
        )

        outputs = {}

        for width in REGION_WIDTHS:
            outputs[
                width
            ] = (
                self.heads[
                    str(width)
                ](
                    target_features,
                    width,
                )
            )

        return outputs


def create_model():
    return BaseLevelRegionalModel()


model_cpu = create_model()

trainable_params = sum(
    p.numel()
    for p in model_cpu.parameters()
    if p.requires_grad
)

frozen_params = sum(
    p.numel()
    for p in model_cpu.parameters()
    if not p.requires_grad
)

print(
    "Backbone channels:",
    model_cpu.backbone_channels,
)

print(
    "Backbone theoretical RF:",
    BACKBONE_RF_BP,
    "bp",
)

print(
    "32-bp regional union RF:",
    REGIONAL_UNION_RF_BP,
    "bp",
)

print(
    "Trainable params:",
    f"{trainable_params:,}",
)

print(
    "Frozen old-head params:",
    f"{frozen_params:,}",
)

del model_cpu


Backbone channels: 32
Backbone theoretical RF: 2053 bp
32-bp regional union RF: 2084 bp
Trainable params: 67,521
Frozen old-head params: 66


In [5]:
# ============================================================
# TARGETS / LOSS / FORWARD HELPERS
# ============================================================


def make_region_targets_cpu(
    exact_target,
    base_weight,
    width,
):
    if (
        exact_target.device.type
        != "cpu"
    ):
        raise ValueError(
            "Expected CPU exact_target"
        )

    if (
        base_weight.device.type
        != "cpu"
    ):
        raise ValueError(
            "Expected CPU base_weight"
        )

    B = exact_target.shape[0]

    n_regions = (
        TARGET_LENGTH
        // width
    )

    target = (
        exact_target
        .float()
        .reshape(
            B,
            1,
            n_regions,
            width,
        )
        .amax(
            dim=-1
        )
    )

    valid = (
        (
            base_weight > 0
        )
        .reshape(
            B,
            1,
            n_regions,
            width,
        )
        .all(
            dim=-1
        )
        .float()
    )

    return (
        target.contiguous(),
        valid.contiguous(),
    )


def stable_region_bce(
    logits,
    targets,
    valid,
    *,
    max_pos_weight=32.0,
):
    targets = targets.to(
        dtype=logits.dtype
    )

    valid = valid.to(
        dtype=logits.dtype
    )

    with torch.no_grad():
        pos = (
            targets * valid
        ).sum()

        neg = (
            (1.0 - targets)
            * valid
        ).sum()

        pos_weight = torch.clamp(
            neg
            / torch.clamp(
                pos,
                min=1.0,
            ),
            min=1.0,
            max=max_pos_weight,
        )

    positive_term = (
        torch.clamp(
            -logits,
            min=0,
        )
        + torch.log1p(
            torch.exp(
                -torch.abs(
                    logits
                )
            )
        )
    )

    negative_term = (
        torch.clamp(
            logits,
            min=0,
        )
        + torch.log1p(
            torch.exp(
                -torch.abs(
                    logits
                )
            )
        )
    )

    element_loss = (
        targets
        * pos_weight
        * positive_term
        + (
            1.0 - targets
        )
        * negative_term
    )

    denom = torch.clamp(
        valid.sum(),
        min=1.0,
    )

    loss = (
        element_loss
        * valid
    ).sum() / denom

    return (
        loss,
        float(
            pos_weight
            .detach()
            .cpu()
            .item()
        ),
    )


def forward_both(
    model,
    x_both,
    batch_size,
):
    outputs_all = model(
        x_both
    )

    result = {}

    for width in REGION_WIDTHS:
        logits = outputs_all[
            width
        ]

        result[
            width
        ] = (
            logits[
                :batch_size
            ].contiguous(),
            logits[
                batch_size:
            ].contiguous(),
        )

    return result


In [6]:
# ============================================================
# FULL HOLDOUT TILE INDEX
# ============================================================

validation_tiles = (
    validation_generator
    .tile_index
    .copy()
)

print(
    "Tiles:",
    len(
        validation_tiles
    ),
)

print(
    "Position-strand:",
    int(
        validation_tiles[
            "allowed_position_strand"
        ].sum()
    ),
)

print(
    "Positives:",
    int(
        validation_tiles[
            "positive_positions"
        ].sum()
    ),
)

print(
    "Contigs:",
    validation_tiles[
        "contig"
    ].nunique(),
)


Tiles: 24700
Position-strand: 83807486
Positives: 56205
Contigs: 3


In [7]:
# ============================================================
# LOAD EXP033 @10K — ANALYSIS ONLY
# ============================================================

RUN_DIR = (
    PROJECT_ROOT
    / "artifacts"
    / "experiments"
    / "CNN_EXP033_BASELEVEL_RF2053_REGIONAL_32_ONLY"
    / "run_001"
)

CHECKPOINT_PATH = (
    RUN_DIR
    / "checkpoint_step_010000.pt"
)

if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(
        CHECKPOINT_PATH
    )

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False,
)

assert (
    checkpoint[
        "experiment"
    ]
    == "CNN-EXP-033"
)

assert (
    checkpoint[
        "architecture"
    ]
    == "baselevel_rf2053_dilated_cnn_regional_32_only"
)

validation_model = (
    create_model()
    .to(device)
)

validation_model.load_state_dict(
    checkpoint[
        "model_state_dict"
    ]
)

validation_model.eval()

print("Loaded:", CHECKPOINT_PATH)
print("Step:", checkpoint["step"])


Loaded: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP033_BASELEVEL_RF2053_REGIONAL_32_ONLY\run_001\checkpoint_step_010000.pt
Step: 10000


In [8]:
# ============================================================
# SCORE FULL HOLDOUT + KEEP POSITIVE-REGION DIAGNOSTICS
# ============================================================

REGION_WIDTH = 32
N_REGIONS = TARGET_LENGTH // REGION_WIDTH
EVAL_BATCH_SIZE = 8


def sigmoid_numpy(
    x,
):
    x = np.asarray(
        x,
        dtype=np.float64,
    )

    out = np.empty_like(
        x
    )

    pos = x >= 0

    out[
        pos
    ] = (
        1.0
        / (
            1.0
            + np.exp(
                -x[
                    pos
                ]
            )
        )
    )

    exp_x = np.exp(
        x[
            ~pos
        ]
    )

    out[
        ~pos
    ] = (
        exp_x
        / (
            1.0
            + exp_x
        )
    )

    return out


score_parts = []
target_parts = []
tss_count_parts = []
read_sum_parts = []
read_max_parts = []
closest_center_parts = []
dominant_offset_parts = []
dominant_edge_parts = []
tile_parts = []
strand_parts = []
region_parts = []

started = time.perf_counter()
positions = 0

with warnings.catch_warnings(
    record=True
) as caught:

    warnings.simplefilter(
        "always"
    )

    with torch.no_grad():
        for batch_no, start in enumerate(
            range(
                0,
                len(validation_tiles),
                EVAL_BATCH_SIZE,
            ),
            1,
        ):
            selected = (
                validation_tiles.iloc[
                    start:
                    start
                    + EVAL_BATCH_SIZE
                ]
            )

            batch = (
                validation_generator
                .make_batch(
                    selected
                )
            )

            B = len(
                selected
            )

            x_gpu = (
                batch
                .x_both_strands
                .to(device)
            )

            outputs = (
                forward_both(
                    validation_model,
                    x_gpu,
                    batch_size=B,
                )
            )

            plus_logits, minus_logits_reverse = (
                outputs[
                    REGION_WIDTH
                ]
            )

            plus_score = sigmoid_numpy(
                plus_logits
                .detach()
                .cpu()
                .numpy()
            )[
                :,
                0,
                :,
            ]

            minus_score = sigmoid_numpy(
                minus_logits_reverse
                .detach()
                .cpu()
                .numpy()
            )[
                :,
                0,
                ::-1,
            ]

            score = np.stack(
                (
                    plus_score,
                    minus_score,
                ),
                axis=1,
            )

            mask = (
                batch
                .mask
                .numpy()
                .astype(bool)
            )

            counts = (
                batch
                .count
                .numpy()
            )

            mask_regions = (
                mask.reshape(
                    B,
                    2,
                    N_REGIONS,
                    REGION_WIDTH,
                )
            )

            count_regions = (
                counts.reshape(
                    B,
                    2,
                    N_REGIONS,
                    REGION_WIDTH,
                )
            )

            valid = (
                mask_regions
                .all(
                    axis=-1
                )
            )

            positive_base = (
                (count_regions > 0)
                & mask_regions
            )

            tss_count = (
                positive_base
                .sum(
                    axis=-1
                )
                .astype(
                    np.int16
                )
            )

            positive = (
                tss_count > 0
            )

            read_sum = (
                (
                    count_regions
                    * mask_regions
                )
                .sum(
                    axis=-1
                )
                .astype(
                    np.float64
                )
            )

            read_max = (
                np.where(
                    mask_regions,
                    count_regions,
                    0,
                )
                .max(
                    axis=-1
                )
                .astype(
                    np.float64
                )
            )

            local_tile, strand, region = (
                np.nonzero(
                    valid
                )
            )

            # Region-internal localization diagnostics.
            n_valid = len(
                local_tile
            )

            closest_center = np.full(
                n_valid,
                np.nan,
                dtype=np.float32,
            )

            dominant_offset = np.full(
                n_valid,
                np.nan,
                dtype=np.float32,
            )

            dominant_edge = np.full(
                n_valid,
                np.nan,
                dtype=np.float32,
            )

            center = (
                REGION_WIDTH
                / 2.0
                - 0.5
            )

            for j in range(
                n_valid
            ):
                b = int(
                    local_tile[
                        j
                    ]
                )

                s = int(
                    strand[
                        j
                    ]
                )

                r = int(
                    region[
                        j
                    ]
                )

                if not positive[
                    b,
                    s,
                    r,
                ]:
                    continue

                region_counts = (
                    count_regions[
                        b,
                        s,
                        r,
                    ]
                )

                positive_positions = (
                    np.flatnonzero(
                        region_counts
                        > 0
                    )
                )

                closest_center[
                    j
                ] = float(
                    np.min(
                        np.abs(
                            positive_positions
                            - center
                        )
                    )
                )

                dominant_position = int(
                    np.argmax(
                        region_counts
                    )
                )

                dominant_offset[
                    j
                ] = float(
                    dominant_position
                    - center
                )

                dominant_edge[
                    j
                ] = float(
                    min(
                        dominant_position,
                        REGION_WIDTH
                        - 1
                        - dominant_position,
                    )
                )

            score_parts.append(
                score[
                    local_tile,
                    strand,
                    region,
                ].astype(
                    np.float32
                )
            )

            target_parts.append(
                positive[
                    local_tile,
                    strand,
                    region,
                ].astype(
                    np.bool_
                )
            )

            tss_count_parts.append(
                tss_count[
                    local_tile,
                    strand,
                    region,
                ]
            )

            read_sum_parts.append(
                read_sum[
                    local_tile,
                    strand,
                    region,
                ]
            )

            read_max_parts.append(
                read_max[
                    local_tile,
                    strand,
                    region,
                ]
            )

            closest_center_parts.append(
                closest_center
            )

            dominant_offset_parts.append(
                dominant_offset
            )

            dominant_edge_parts.append(
                dominant_edge
            )

            tile_parts.append(
                (
                    start
                    + local_tile
                ).astype(
                    np.int32
                )
            )

            strand_parts.append(
                strand.astype(
                    np.int8
                )
            )

            region_parts.append(
                region.astype(
                    np.int16
                )
            )

            positions += int(
                mask.sum()
            )

            finished = (
                start + B
                >= len(
                    validation_tiles
                )
            )

            if (
                batch_no == 1
                or batch_no % 500 == 0
                or finished
            ):
                elapsed = (
                    time.perf_counter()
                    - started
                )

                print(
                    f"tiles="
                    f"{start + B:>5}"
                    f"/{len(validation_tiles)} "
                    f"positions/s="
                    f"{positions / elapsed:,.0f}"
                )

fallback = [
    str(w.message)
    for w in caught
    if "fall back"
    in str(
        w.message
    ).lower()
]

if fallback:
    raise RuntimeError(
        "DirectML fallback: "
        + " | ".join(
            fallback
        )
    )

scores_all = np.concatenate(
    score_parts
)

targets_all = np.concatenate(
    target_parts
)

tss_count_all = np.concatenate(
    tss_count_parts
)

read_sum_all = np.concatenate(
    read_sum_parts
)

read_max_all = np.concatenate(
    read_max_parts
)

closest_center_all = np.concatenate(
    closest_center_parts
)

dominant_offset_all = np.concatenate(
    dominant_offset_parts
)

dominant_edge_all = np.concatenate(
    dominant_edge_parts
)

tile_rows_all = np.concatenate(
    tile_parts
)

strands_all = np.concatenate(
    strand_parts
)

regions_all = np.concatenate(
    region_parts
)

print()
print(
    "Valid regions:",
    f"{len(scores_all):,}",
)

print(
    "Positive regions:",
    f"{int(targets_all.sum()):,}",
)

print(
    "TSS positions:",
    f"{int(tss_count_all.sum()):,}",
)

print(
    "Total reads in positive regions:",
    f"{read_sum_all[targets_all].sum():,.0f}",
)


tiles=    8/24700 positions/s=107,772
tiles= 4000/24700 positions/s=2,528,725
tiles= 8000/24700 positions/s=2,497,932
tiles=12000/24700 positions/s=2,425,104
tiles=16000/24700 positions/s=2,416,808
tiles=20000/24700 positions/s=2,341,420
tiles=24000/24700 positions/s=2,370,062
tiles=24700/24700 positions/s=2,367,831

Valid regions: 2,560,380
Positive regions: 27,982
TSS positions: 55,730
Total reads in positive regions: 486,766


In [9]:
# ============================================================
# GLOBAL TOP-10% THRESHOLD + POSITIVE REGION TABLE
# ============================================================

TOP_FRACTION = 0.10

n_top = int(
    np.ceil(
        TOP_FRACTION
        * len(
            scores_all
        )
    )
)

global_order = np.argsort(
    -scores_all,
    kind="mergesort",
)

selected_top10 = np.zeros(
    len(
        scores_all
    ),
    dtype=bool,
)

selected_top10[
    global_order[
        :n_top
    ]
] = True

positive_idx = np.flatnonzero(
    targets_all
)

positive_regions = pd.DataFrame({
    "catalog_index": (
        positive_idx
    ),
    "score": (
        scores_all[
            positive_idx
        ]
    ),
    "selected_top10": (
        selected_top10[
            positive_idx
        ]
    ),
    "tss_count": (
        tss_count_all[
            positive_idx
        ]
    ),
    "read_sum": (
        read_sum_all[
            positive_idx
        ]
    ),
    "read_max": (
        read_max_all[
            positive_idx
        ]
    ),
    "closest_tss_abs_offset_to_region_center": (
        closest_center_all[
            positive_idx
        ]
    ),
    "dominant_tss_signed_offset_from_center": (
        dominant_offset_all[
            positive_idx
        ]
    ),
    "dominant_tss_distance_to_nearest_edge": (
        dominant_edge_all[
            positive_idx
        ]
    ),
    "tile_row": (
        tile_rows_all[
            positive_idx
        ]
    ),
    "strand_index": (
        strands_all[
            positive_idx
        ]
    ),
    "region_index": (
        regions_all[
            positive_idx
        ]
    ),
})

positive_regions[
    "strand"
] = np.where(
    positive_regions[
        "strand_index"
    ]
    == 0,
    "+",
    "-",
)

positive_regions[
    "log1p_read_sum"
] = np.log1p(
    positive_regions[
        "read_sum"
    ]
)

positive_regions[
    "log1p_read_max"
] = np.log1p(
    positive_regions[
        "read_max"
    ]
)

capture_summary = pd.DataFrame([
    {
        "group": "CAPTURED_TOP10",
        "positive_regions": int(
            positive_regions[
                "selected_top10"
            ].sum()
        ),
        "fraction_positive_regions": float(
            positive_regions[
                "selected_top10"
            ].mean()
        ),
        "tss_positions": int(
            positive_regions.loc[
                positive_regions[
                    "selected_top10"
                ],
                "tss_count",
            ].sum()
        ),
        "fraction_tss_positions": float(
            positive_regions.loc[
                positive_regions[
                    "selected_top10"
                ],
                "tss_count",
            ].sum()
            / positive_regions[
                "tss_count"
            ].sum()
        ),
        "reads": float(
            positive_regions.loc[
                positive_regions[
                    "selected_top10"
                ],
                "read_sum",
            ].sum()
        ),
        "fraction_reads": float(
            positive_regions.loc[
                positive_regions[
                    "selected_top10"
                ],
                "read_sum",
            ].sum()
            / positive_regions[
                "read_sum"
            ].sum()
        ),
    },
    {
        "group": "MISSED_TOP10",
        "positive_regions": int(
            (
                ~positive_regions[
                    "selected_top10"
                ]
            ).sum()
        ),
        "fraction_positive_regions": float(
            (
                ~positive_regions[
                    "selected_top10"
                ]
            ).mean()
        ),
        "tss_positions": int(
            positive_regions.loc[
                ~positive_regions[
                    "selected_top10"
                ],
                "tss_count",
            ].sum()
        ),
        "fraction_tss_positions": float(
            positive_regions.loc[
                ~positive_regions[
                    "selected_top10"
                ],
                "tss_count",
            ].sum()
            / positive_regions[
                "tss_count"
            ].sum()
        ),
        "reads": float(
            positive_regions.loc[
                ~positive_regions[
                    "selected_top10"
                ],
                "read_sum",
            ].sum()
        ),
        "fraction_reads": float(
            positive_regions.loc[
                ~positive_regions[
                    "selected_top10"
                ],
                "read_sum",
            ].sum()
            / positive_regions[
                "read_sum"
            ].sum()
        ),
    },
])

display(
    capture_summary
)

print(
    "Global TOP-10% score boundary:",
    float(
        scores_all[
            global_order[
                n_top - 1
            ]
        ]
    ),
)


,group,positive_regions,fraction_positive_regions,tss_positions,fraction_tss_positions,reads,fraction_reads
0,CAPTURED_TOP10,16509,0.589986,39960,0.717029,386403.0,0.793817
1,MISSED_TOP10,11473,0.410014,15770,0.282971,100363.0,0.206183


Global TOP-10% score boundary: 0.7552434206008911


In [10]:
# ============================================================
# DOES MODEL SCORE MAINLY TRACK TSS STRENGTH?
# ============================================================

strength_correlations = pd.DataFrame([
    {
        "x": "score",
        "y": "read_sum",
        "spearman": float(
            positive_regions[
                [
                    "score",
                    "read_sum",
                ]
            ]
            .corr(
                method="spearman"
            )
            .iloc[
                0,
                1,
            ]
        ),
    },
    {
        "x": "score",
        "y": "read_max",
        "spearman": float(
            positive_regions[
                [
                    "score",
                    "read_max",
                ]
            ]
            .corr(
                method="spearman"
            )
            .iloc[
                0,
                1,
            ]
        ),
    },
    {
        "x": "score",
        "y": "tss_count",
        "spearman": float(
            positive_regions[
                [
                    "score",
                    "tss_count",
                ]
            ]
            .corr(
                method="spearman"
            )
            .iloc[
                0,
                1,
            ]
        ),
    },
])

display(
    strength_correlations
)

READ_BINS = (
    0,
    1,
    2,
    4,
    9,
    24,
    99,
    np.inf,
)

READ_LABELS = (
    "1",
    "2",
    "3-4",
    "5-9",
    "10-24",
    "25-99",
    "100+",
)

positive_regions[
    "read_strength_bin"
] = pd.cut(
    positive_regions[
        "read_sum"
    ],
    bins=READ_BINS,
    labels=READ_LABELS,
    include_lowest=True,
)

strength_capture = (
    positive_regions
    .groupby(
        "read_strength_bin",
        observed=True,
        as_index=False,
    )
    .agg(
        positive_regions=(
            "score",
            "size",
        ),
        median_score=(
            "score",
            "median",
        ),
        top10_capture_rate=(
            "selected_top10",
            "mean",
        ),
        median_tss_count=(
            "tss_count",
            "median",
        ),
    )
)

display(
    strength_capture
)


,x,y,spearman
0,score,read_sum,0.336881
1,score,read_max,0.297342
2,score,tss_count,0.362785


,read_strength_bin,positive_regions,median_score,top10_capture_rate,median_tss_count
0,1,13611,0.751171,0.494673,1.0
1,2,5194,0.791942,0.554101,1.0
2,3-4,3520,0.837185,0.632386,2.0
3,5-9,2221,0.902444,0.755966,3.0
4,10-24,1562,0.937174,0.839309,4.0
5,25-99,1220,0.958590,0.881148,7.0
6,100+,654,0.979253,0.928135,11.0


In [11]:
# ============================================================
# LOCALIZATION INSIDE THE 32-BP BIN
# ============================================================

localization_summary = (
    positive_regions
    .assign(
        capture_group=np.where(
            positive_regions[
                "selected_top10"
            ],
            "CAPTURED_TOP10",
            "MISSED_TOP10",
        )
    )
    .groupby(
        "capture_group",
        as_index=False,
    )
    .agg(
        n=(
            "score",
            "size",
        ),
        score_median=(
            "score",
            "median",
        ),
        read_sum_median=(
            "read_sum",
            "median",
        ),
        tss_count_median=(
            "tss_count",
            "median",
        ),
        closest_tss_to_center_median_bp=(
            "closest_tss_abs_offset_to_region_center",
            "median",
        ),
        dominant_tss_edge_distance_median_bp=(
            "dominant_tss_distance_to_nearest_edge",
            "median",
        ),
        dominant_tss_signed_offset_median_bp=(
            "dominant_tss_signed_offset_from_center",
            "median",
        ),
    )
)

display(
    localization_summary
)

# Exact dominant-TSS position histogram inside a 32-bp region.
position_table = (
    positive_regions
    .assign(
        dominant_position=(
            positive_regions[
                "dominant_tss_signed_offset_from_center"
            ]
            + (
                REGION_WIDTH
                / 2.0
                - 0.5
            )
        ).round().astype(int),
        capture_group=np.where(
            positive_regions[
                "selected_top10"
            ],
            "CAPTURED_TOP10",
            "MISSED_TOP10",
        ),
    )
    .groupby(
        [
            "capture_group",
            "dominant_position",
        ],
        as_index=False,
    )
    .size()
)

position_table[
    "fraction"
] = (
    position_table[
        "size"
    ]
    / position_table.groupby(
        "capture_group"
    )[
        "size"
    ]
    .transform(
        "sum"
    )
)

display(
    position_table
    .pivot(
        index="dominant_position",
        columns="capture_group",
        values="fraction",
    )
)


,capture_group,n,score_median,read_sum_median,tss_count_median,closest_tss_to_center_median_bp,dominant_tss_edge_distance_median_bp,dominant_tss_signed_offset_median_bp
0,CAPTURED_TOP10,16509,0.914254,2.0,1.0,5.5,7.0,-0.5
1,MISSED_TOP10,11473,0.511129,1.0,1.0,7.5,7.0,-0.5


capture_group,CAPTURED_TOP10,MISSED_TOP10
dominant_position,,
0,0.039736,0.038961
1,0.037192,0.039571
2,0.036465,0.037218
3,0.036223,0.034080
4,0.033860,0.033383
5,0.033073,0.032860
6,0.033376,0.032075
7,0.029257,0.033034
8,0.031377,0.029896


In [12]:
# ============================================================
# DEFINE HIGH_TP / LOW_TP EXTREMES
# ============================================================

EXTREME_N = 8_000

positive_order = (
    positive_idx[
        np.argsort(
            -scores_all[
                positive_idx
            ],
            kind="mergesort",
        )
    ]
)

high_tp_idx = (
    positive_order[
        :EXTREME_N
    ]
)

low_tp_idx = (
    positive_order[
        -EXTREME_N:
    ][::-1]
)

extreme_summary = pd.DataFrame([
    {
        "group": "HIGH_TP",
        "n": len(
            high_tp_idx
        ),
        "score_median": float(
            np.median(
                scores_all[
                    high_tp_idx
                ]
            )
        ),
        "read_sum_median": float(
            np.median(
                read_sum_all[
                    high_tp_idx
                ]
            )
        ),
        "read_max_median": float(
            np.median(
                read_max_all[
                    high_tp_idx
                ]
            )
        ),
        "tss_count_median": float(
            np.median(
                tss_count_all[
                    high_tp_idx
                ]
            )
        ),
    },
    {
        "group": "LOW_TP",
        "n": len(
            low_tp_idx
        ),
        "score_median": float(
            np.median(
                scores_all[
                    low_tp_idx
                ]
            )
        ),
        "read_sum_median": float(
            np.median(
                read_sum_all[
                    low_tp_idx
                ]
            )
        ),
        "read_max_median": float(
            np.median(
                read_max_all[
                    low_tp_idx
                ]
            )
        ),
        "tss_count_median": float(
            np.median(
                tss_count_all[
                    low_tp_idx
                ]
            )
        ),
    },
])

display(
    extreme_summary
)


,group,n,score_median,read_sum_median,read_max_median,tss_count_median
0,HIGH_TP,8000,0.964893,3.0,2.0,2.0
1,LOW_TP,8000,0.404294,1.0,1.0,1.0


In [13]:
# ============================================================
# EXTRACT 256-BP STRAND-ORIENTED DNA FOR HIGH_TP / LOW_TP
# ============================================================

CONTEXT_BP = 256
HALF_CONTEXT = CONTEXT_BP // 2

assert (
    HALF_CONTEXT
    <= CONTEXT_FLANK
    + REGION_WIDTH // 2
)

BASES = np.array(
    list(
        "ACGTN"
    )
)


def catalog_frame(
    catalog_indices,
    group_name,
):
    catalog_indices = np.asarray(
        catalog_indices,
        dtype=np.int64,
    )

    return pd.DataFrame({
        "group": group_name,
        "catalog_index": (
            catalog_indices
        ),
        "score": (
            scores_all[
                catalog_indices
            ]
        ),
        "tss_count": (
            tss_count_all[
                catalog_indices
            ]
        ),
        "read_sum": (
            read_sum_all[
                catalog_indices
            ]
        ),
        "read_max": (
            read_max_all[
                catalog_indices
            ]
        ),
        "tile_row": (
            tile_rows_all[
                catalog_indices
            ]
        ),
        "strand_index": (
            strands_all[
                catalog_indices
            ]
        ),
        "region_index": (
            regions_all[
                catalog_indices
            ]
        ),
    })


extreme_sequences = pd.concat(
    (
        catalog_frame(
            high_tp_idx,
            "HIGH_TP",
        ),
        catalog_frame(
            low_tp_idx,
            "LOW_TP",
        ),
    ),
    ignore_index=True,
)

extreme_sequences[
    "seq_core32"
] = ""

extreme_sequences[
    "seq_context256"
] = ""

tile_groups = (
    extreme_sequences
    .groupby(
        "tile_row"
    )
    .indices
)

unique_tile_rows = np.array(
    sorted(
        tile_groups.keys()
    ),
    dtype=np.int64,
)

EXTRACT_BATCH_SIZE = 64

for batch_start in range(
    0,
    len(
        unique_tile_rows
    ),
    EXTRACT_BATCH_SIZE,
):
    tile_rows = (
        unique_tile_rows[
            batch_start:
            batch_start
            + EXTRACT_BATCH_SIZE
        ]
    )

    batch = (
        validation_generator
        .make_batch(
            validation_tiles.iloc[
                tile_rows
            ]
        )
    )

    x = (
        batch
        .x_both_strands
        .numpy()
    )

    B = len(
        tile_rows
    )

    tile_to_local = {
        int(tile_row): local
        for local, tile_row
        in enumerate(
            tile_rows
        )
    }

    for tile_row in tile_rows:
        local = (
            tile_to_local[
                int(
                    tile_row
                )
            ]
        )

        for row_idx in (
            tile_groups[
                int(
                    tile_row
                )
            ]
        ):
            strand_index = int(
                extreme_sequences.at[
                    row_idx,
                    "strand_index",
                ]
            )

            genomic_region = int(
                extreme_sequences.at[
                    row_idx,
                    "region_index",
                ]
            )

            if strand_index == 0:
                oriented_row = (
                    local
                )

                oriented_region = (
                    genomic_region
                )
            else:
                oriented_row = (
                    B + local
                )

                oriented_region = (
                    N_REGIONS
                    - 1
                    - genomic_region
                )

            core_start = (
                CONTEXT_FLANK
                + oriented_region
                * REGION_WIDTH
            )

            core_end = (
                core_start
                + REGION_WIDTH
            )

            center = (
                core_start
                + REGION_WIDTH // 2
            )

            context_start = (
                center
                - HALF_CONTEXT
            )

            context_end = (
                center
                + HALF_CONTEXT
            )

            core_codes = np.argmax(
                x[
                    oriented_row,
                    :,
                    core_start:
                    core_end,
                ],
                axis=0,
            )

            context_codes = np.argmax(
                x[
                    oriented_row,
                    :,
                    context_start:
                    context_end,
                ],
                axis=0,
            )

            extreme_sequences.at[
                row_idx,
                "seq_core32",
            ] = "".join(
                BASES[
                    core_codes
                ]
            )

            extreme_sequences.at[
                row_idx,
                "seq_context256",
            ] = "".join(
                BASES[
                    context_codes
                ]
            )

print(
    "Extracted sequences:",
    len(
        extreme_sequences
    ),
)

assert (
    extreme_sequences[
        "seq_context256"
    ]
    .str.len()
    .eq(
        CONTEXT_BP
    )
    .all()
)


Extracted sequences: 16000


In [14]:
# ============================================================
# COMPOSITION HIGH_TP vs LOW_TP
# ============================================================

def sequence_stats(
    seq,
):
    counts = {
        b: seq.count(
            b
        )
        for b in "ACGTN"
    }

    valid = sum(
        counts[
            b
        ]
        for b in "ACGT"
    )

    gc = (
        (
            counts[
                "G"
            ]
            + counts[
                "C"
            ]
        )
        / valid
        if valid > 0
        else np.nan
    )

    probs = np.array(
        [
            counts[
                b
            ]
            for b in "ACGT"
        ],
        dtype=np.float64,
    )

    if probs.sum() > 0:
        probs /= probs.sum()

        nz = (
            probs > 0
        )

        entropy = float(
            -np.sum(
                probs[
                    nz
                ]
                * np.log2(
                    probs[
                        nz
                    ]
                )
            )
        )
    else:
        entropy = np.nan

    return (
        gc,
        counts[
            "N"
        ]
        / len(
            seq
        ),
        entropy,
    )


core_stats = np.array(
    [
        sequence_stats(
            seq
        )
        for seq in (
            extreme_sequences[
                "seq_core32"
            ]
        )
    ]
)

context_stats = np.array(
    [
        sequence_stats(
            seq
        )
        for seq in (
            extreme_sequences[
                "seq_context256"
            ]
        )
    ]
)

extreme_sequences[
    "gc_core32"
] = core_stats[
    :,
    0,
]

extreme_sequences[
    "entropy_core32"
] = core_stats[
    :,
    2,
]

extreme_sequences[
    "gc_context256"
] = context_stats[
    :,
    0,
]

extreme_sequences[
    "entropy_context256"
] = context_stats[
    :,
    2,
]

composition_summary = (
    extreme_sequences
    .groupby(
        "group",
        as_index=False,
    )
    .agg(
        n=(
            "score",
            "size",
        ),
        score_median=(
            "score",
            "median",
        ),
        read_sum_median=(
            "read_sum",
            "median",
        ),
        gc_core32_mean=(
            "gc_core32",
            "mean",
        ),
        gc_context256_mean=(
            "gc_context256",
            "mean",
        ),
        entropy_core32_mean=(
            "entropy_core32",
            "mean",
        ),
        entropy_context256_mean=(
            "entropy_context256",
            "mean",
        ),
    )
)

display(
    composition_summary
)


,group,n,score_median,read_sum_median,gc_core32_mean,gc_context256_mean,entropy_core32_mean,entropy_context256_mean
0,HIGH_TP,8000,0.964893,3.0,0.460016,0.435533,1.904586,1.968801
1,LOW_TP,8000,0.404294,1.0,0.417160,0.407799,1.883353,1.952001


In [15]:
# ============================================================
# 4-MER + POSITION-AWARE PROBES: HIGH_TP vs LOW_TP
# ============================================================

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

BASE_TO_INT = {
    "A": 0,
    "C": 1,
    "G": 2,
    "T": 3,
}


def kmer_vector(
    seq,
    k=4,
):
    vector = np.zeros(
        4 ** k,
        dtype=np.float32,
    )

    for i in range(
        len(seq)
        - k
        + 1
    ):
        value = 0
        valid = True

        for char in seq[
            i:
            i + k
        ]:
            code = (
                BASE_TO_INT
                .get(
                    char
                )
            )

            if code is None:
                valid = False
                break

            value = (
                value * 4
                + code
            )

        if valid:
            vector[
                value
            ] += 1.0

    total = (
        vector.sum()
    )

    if total > 0:
        vector /= total

    return vector


def position_onehot(
    seq,
):
    vector = np.zeros(
        CONTEXT_BP
        * 4,
        dtype=np.float32,
    )

    for pos, char in enumerate(
        seq
    ):
        code = (
            BASE_TO_INT
            .get(
                char
            )
        )

        if code is not None:
            vector[
                pos * 4
                + code
            ] = 1.0

    return vector


group_array = (
    extreme_sequences[
        "group"
    ]
    .to_numpy()
)

y = (
    group_array
    == "LOW_TP"
).astype(
    np.int8
)

pair_id = np.concatenate(
    (
        np.arange(
            EXTREME_N
        ),
        np.arange(
            EXTREME_N
        ),
    )
)

train_pair, test_pair = (
    train_test_split(
        np.arange(
            EXTREME_N
        ),
        test_size=0.30,
        random_state=42,
    )
)

train_mask = np.isin(
    pair_id,
    train_pair,
)

test_mask = np.isin(
    pair_id,
    test_pair,
)


def fit_auc(
    X,
):
    clf = LogisticRegression(
        max_iter=1500,
        C=1.0,
        solver="liblinear",
        random_state=42,
    )

    clf.fit(
        X[
            train_mask
        ],
        y[
            train_mask
        ],
    )

    prob = (
        clf
        .predict_proba(
            X[
                test_mask
            ]
        )[
            :,
            1
        ]
    )

    return float(
        roc_auc_score(
            y[
                test_mask
            ],
            prob,
        )
    )


X4 = np.stack(
    [
        kmer_vector(
            seq
        )
        for seq in (
            extreme_sequences[
                "seq_context256"
            ]
        )
    ]
)

Xpos = np.stack(
    [
        position_onehot(
            seq
        )
        for seq in (
            extreme_sequences[
                "seq_context256"
            ]
        )
    ]
)

probe_results = pd.DataFrame([
    {
        "comparison": "LOW_TP vs HIGH_TP",
        "feature_set": "bag_of_4mer_256bp",
        "roc_auc": fit_auc(
            X4
        ),
    },
    {
        "comparison": "LOW_TP vs HIGH_TP",
        "feature_set": "position_aware_base_256bp",
        "roc_auc": fit_auc(
            Xpos
        ),
    },
])

display(
    probe_results
)


,comparison,feature_set,roc_auc
0,LOW_TP vs HIGH_TP,bag_of_4mer_256bp,0.733220
1,LOW_TP vs HIGH_TP,position_aware_base_256bp,0.694433


In [16]:
# ============================================================
# 6-MER DIFFERENCES: LOW_TP vs HIGH_TP
# ============================================================

from collections import Counter

K = 6
PSEUDOCOUNT = 1.0
VOCAB = 4 ** K


def aggregate_kmers(
    sequences,
    k,
):
    counter = Counter()
    total = 0

    for seq in sequences:
        for i in range(
            len(seq)
            - k
            + 1
        ):
            kmer = (
                seq[
                    i:
                    i + k
                ]
            )

            if "N" in kmer:
                continue

            counter[
                kmer
            ] += 1

            total += 1

    return (
        counter,
        total,
    )


high_sequences = (
    extreme_sequences.loc[
        extreme_sequences[
            "group"
        ]
        == "HIGH_TP",
        "seq_context256",
    ]
    .tolist()
)

low_sequences = (
    extreme_sequences.loc[
        extreme_sequences[
            "group"
        ]
        == "LOW_TP",
        "seq_context256",
    ]
    .tolist()
)

high_counter, high_total = (
    aggregate_kmers(
        high_sequences,
        K,
    )
)

low_counter, low_total = (
    aggregate_kmers(
        low_sequences,
        K,
    )
)

all_kmers = sorted(
    set(
        high_counter
    )
    | set(
        low_counter
    )
)

rows = []

for kmer in all_kmers:
    high_freq = (
        high_counter[
            kmer
        ]
        + PSEUDOCOUNT
    ) / (
        high_total
        + PSEUDOCOUNT
        * VOCAB
    )

    low_freq = (
        low_counter[
            kmer
        ]
        + PSEUDOCOUNT
    ) / (
        low_total
        + PSEUDOCOUNT
        * VOCAB
    )

    rows.append({
        "kmer": kmer,
        "log2_LOW_vs_HIGH": (
            np.log2(
                low_freq
                / high_freq
            )
        ),
        "frequency_LOW": (
            low_freq
        ),
        "frequency_HIGH": (
            high_freq
        ),
    })

kmer_differences = (
    pd.DataFrame(
        rows
    )
)

print(
    "=== MOST LOW_TP-ENRICHED 6-MERS ==="
)

display(
    kmer_differences
    .nlargest(
        30,
        "log2_LOW_vs_HIGH",
    )
)

print()
print(
    "=== MOST HIGH_TP-ENRICHED 6-MERS ==="
)

display(
    kmer_differences
    .nsmallest(
        30,
        "log2_LOW_vs_HIGH",
    )
)


=== MOST LOW_TP-ENRICHED 6-MERS ===


,kmer,log2_LOW_vs_HIGH,frequency_LOW,frequency_HIGH
2730,GGGGGG,2.021606,0.000603,0.000149
682,AGGGGG,1.117279,0.000274,0.000126
1194,CAGGGG,1.104046,0.000207,0.000096
683,AGGGGT,1.053494,0.000219,0.000105
2729,GGGGGC,1.020665,0.000140,0.000069
2734,GGGGTG,0.996600,0.000207,0.000104
2732,GGGGTA,0.950630,0.000185,0.000096
3114,TAAGGG,0.897844,0.000196,0.000105
3754,TGGGGG,0.897553,0.000209,0.000112
960,ATTAAA,0.878343,0.000627,0.000341



=== MOST HIGH_TP-ENRICHED 6-MERS ===


,kmer,log2_LOW_vs_HIGH,frequency_LOW,frequency_HIGH
934,ATGGCG,-1.789497,0.000214,0.000741
2465,GCGGAC,-1.746013,0.000079,0.000263
3738,TGGCGG,-1.710438,0.000169,0.000555
2664,GGCGGA,-1.525928,0.000116,0.000333
1670,CGGACG,-1.466644,0.000072,0.000198
2584,GGACGA,-1.311889,0.000130,0.000322
2665,GGCGGC,-1.279305,0.000113,0.000274
1671,CGGACT,-1.216867,0.000075,0.000174
1690,CGGCGG,-1.198171,0.000064,0.000147
3718,TGGACG,-1.174678,0.000116,0.000261


In [17]:
# ============================================================
# SAVE LOW_TP STUDY
# ============================================================

OUT_DIR = (
    RUN_DIR
    / "low_tp_study"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

capture_summary.to_csv(
    OUT_DIR
    / "top10_capture_summary.csv",
    index=False,
)

positive_regions.to_csv(
    OUT_DIR
    / "positive_region_catalog.csv",
    index=False,
)

strength_correlations.to_csv(
    OUT_DIR
    / "score_vs_strength_correlations.csv",
    index=False,
)

strength_capture.to_csv(
    OUT_DIR
    / "capture_by_read_strength.csv",
    index=False,
)

localization_summary.to_csv(
    OUT_DIR
    / "localization_summary.csv",
    index=False,
)

position_table.to_csv(
    OUT_DIR
    / "dominant_tss_position_inside_32bp.csv",
    index=False,
)

extreme_summary.to_csv(
    OUT_DIR
    / "high_low_tp_summary.csv",
    index=False,
)

composition_summary.to_csv(
    OUT_DIR
    / "high_low_tp_composition.csv",
    index=False,
)

probe_results.to_csv(
    OUT_DIR
    / "high_low_tp_sequence_probe_auc.csv",
    index=False,
)

kmer_differences.to_csv(
    OUT_DIR
    / "high_low_tp_6mer_differences.csv",
    index=False,
)

extreme_sequences.to_csv(
    OUT_DIR
    / "high_low_tp_sequences.csv",
    index=False,
)

print(
    "Saved to:",
    OUT_DIR,
)


Saved to: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP033_BASELEVEL_RF2053_REGIONAL_32_ONLY\run_001\low_tp_study


# Что прислать после запуска

Сначала достаточно четырёх результатов:

### 1. `capture_summary`

Особенно:

```text
fraction_positive_regions
fraction_tss_positions
fraction_reads
```

Если TOP-10% теряет много TSS positions, но почти не теряет reads — модель в основном
пропускает слабые initiation events.

### 2. `strength_correlations` + `strength_capture`

Покажет, насколько score связан с csRNA-seq strength.

### 3. `localization_summary`

Проверяем, не чаще ли MISSED_TOP10 лежат краем внутри 32-bp bin.

### 4. `probe_results`

```text
bag-of-4mer AUC
position-aware AUC
```

Если HIGH_TP и LOW_TP легко различаются по sequence, это наиболее перспективный путь:
искать отдельную promoter architecture, которую текущая CNN недоучила.
